# Parser / OCR Bake-off - Round R14 Scoring (H146-H151)

**Finisher pass.** CPU scoring only - reuses the predecessor harness (`parser_harness.py`, `score_round.py`) that already parsed every document and cached outputs under `reports/parser-round-cache/`. This notebook re-derives the headline numbers, records honest coverage fractions (the MinerU2.5 VLM crashed on 10 of 11 chunks), assembles the six verdicts against each registered Acceptance bar, and writes `reports/parser-round-final-<UTCstamp>.json`.

Scoring set: the H51 parse-fidelity loss set (95 names pymupdf4llm drops but pdfplumber/pypdf recover), the H144 boundary audit (64 severed table rows), and the R14 parser-round cache.

In [1]:
# Imports and project-root resolution (robust to nbconvert cwd)
import os, sys, json, glob, datetime, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

p = Path.cwd()
while not (p / 'reports' / 'parser-round-cache').exists() and p != p.parent:
    p = p.parent
ROOT = p
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'notebooks'))

import score_round as S  # module body loads cached texts + rebuilds LOSS/absent (CPU only; __main__ not run)
print('ROOT', ROOT)
print('LOSS names', len(S.LOSS), '| absent-from-all-text', len(S.absent), '| SleepStyle family', len(S.ss_family))

ROOT /home/lab/workspace/learning/projects/knowledge-graph-foundry
LOSS names 95 | absent-from-all-text 676 | SleepStyle family 7


## Per-parser loss-set recovery and coverage

Loss-set recovery = fraction of the 95 names present in the parser's output on the credited source doc. Coverage records how much of each parser's intended page set actually produced usable output.

In [2]:
recall_by_parser, preservation_by_parser = {}, {}
for parser in S.ALLP:
    hit, n = S.loss_recovery(parser)
    recall_by_parser[parser] = hit / n
    preservation_by_parser[parser] = S.preservation(parser)
    print(f'{parser:18s} loss-recovery {hit:3d}/{n} = {hit/n:6.1%}   preservation {S.preservation(parser):6.1%}')

# Coverage fractions (honest denominators)
COMBINED_PAGES = 74 + 33            # combined_manifest.json + combined_manifest2.json vision page universe
numeric_pages_total = sum(len(v) for v in S.numeric_pages.values())
cov = {
    'docling':          {'scope': 'full-doc', 'docs': f'{len(S.load_text("docling"))}/27 docs'},
    'mineru2.5':        {'vision_pages': f'{len(S.mineru_pp)}/{COMBINED_PAGES}',
                          'docs_covered': list((S.mineru_doc or {}).keys()),
                          'numeric_pages': f'{sum(1 for d,pgs in S.numeric_pages.items() for pg in pgs if (d,pg) in (S.mineru_pp or {}))}/{numeric_pages_total}',
                          'note': 'VLM tensor-shape RuntimeError on 10/11 chunks; only chunk_010 (6 usable pages) survived'},
    'olmocr':           {'vision_pages': f'{len(S.olm_off_pp)}/{COMBINED_PAGES}',
                          'numeric_pages': f'{sum(1 for d,pgs in S.numeric_pages.items() for pg in pgs if (d,pg) in S.olm_off_pp)}/{numeric_pages_total}',
                          'note': 'full loss/absent/numeric page coverage, anchored and anchor-off'},
}
print()
print('MinerU2.5 usable vision pages:', cov['mineru2.5']['vision_pages'], '| numeric:', cov['mineru2.5']['numeric_pages'])
print('olmOCR usable vision pages:', cov['olmocr']['vision_pages'], '| numeric:', cov['olmocr']['numeric_pages'])

pymupdf4llm        loss-recovery   0/95 =   0.0%   preservation  72.4%


pdfplumber         loss-recovery  39/95 =  41.1%   preservation  72.1%


pypdf              loss-recovery  89/95 =  93.7%   preservation  75.1%


docling            loss-recovery  82/95 =  86.3%   preservation  73.2%
mineru2.5          loss-recovery   0/95 =   0.0%   preservation   0.6%
olmocr_anchored    loss-recovery  77/95 =  81.1%   preservation  28.8%


olmocr_off         loss-recovery  83/95 =  87.4%   preservation  29.7%

MinerU2.5 usable vision pages: 6/107 | numeric: 0/18
olmOCR usable vision pages: 107/107 | numeric: 18/18


## Load the six per-hypothesis reports and assemble finisher verdicts

The harness wrote one JSON per hypothesis (richer sub-scores: H147 family matrix, H148 floor residue, H150 TEDS correlation). The finisher overrides two verdicts on honest-coverage grounds: **H148 = NOT MEASURABLE** (registered engines have zero numeric-page coverage) and **H151 = PARTIALLY CONFIRMED** (clause a holds, clause b refuted).

In [3]:
def latest(pat):
    fs = sorted(glob.glob(str(ROOT / 'reports' / pat)))
    return json.load(open(fs[-1])) if fs else None

rep = {
    'H146': latest('parser-h146-docling-*.json'),
    'H147': latest('parser-h147-mineru-absent-*.json'),
    'H148': latest('parser-h148-numeric-*.json'),
    'H149': latest('parser-h149-olmocr-anchor-*.json'),
    'H150': latest('parser-h150-teds-corr-*.json'),
    'H151': latest('parser-h151-table-partition-*.json'),
}

h144 = json.load(open(sorted(glob.glob(str(ROOT / 'reports' / 'boundary-audit-h144-*.json')))[-1]))
severed_rows = h144['total_severed_table_rows']

verdicts = {
  'H146': ('CONFIRMED',
    f"Docling recovers {rep['H146']['loss_set_hits']}/{rep['H146']['loss_set_n']} = {rep['H146']['loss_set_recovery']:.1%} of the loss set at {rep['H146']['runtime_s_per_page']} s/page CPU, clearing the >=60% bar and beating the 75.8% union-of-three baseline."),
  'H147': ('REFUTED',
    "The all-parser-absent SleepStyle family is a trademark-glyph normalization artifact - pypdf, docling and olmOCR each recover 7/7 under symbol-stripped matching, so vision is not necessary; MinerU2.5 itself is unmeasurable (VLM crashed on 10/11 chunks, 6/107 pages, 0 family pages covered)."),
  'H148': ('NOT MEASURABLE',
    f"MinerU2.5 covered 0/{numeric_pages_total} numeric-floor pages (VLM crash) and dots.ocr was not run, so the >=10-point recall-gain-with-digit-precision bar cannot be evaluated on the registered engines; the olmOCR-off vision proxy lifted 0/{rep['H148']['floor_residue_n']} of the all-text-parser-failed gold residue, i.e. no vision lift observed."),
  'H149': ('CONFIRMED',
    f"Anchored olmOCR recovers {rep['H149']['anchored_hits']}/{rep['H149']['n']} = {rep['H149']['anchored_recovery']:.1%} (>=half) and anchor-off recovers strictly more at {rep['H149']['anchor_off_hits']}/{rep['H149']['n']} = {rep['H149']['anchor_off_recovery']:.1%} - document-anchoring is a liability on born-digital pages."),
  'H150': ('CONFIRMED',
    f"Spearman(TEDS rank, loss-set name recall) = {rep['H150']['spearman']:.2f} full / {rep['H150']['spearman_bench_only']:.2f} bench-only, far below the 0.5 bar; pypdf (no table stage, TEDS 0) posts the highest recall (93.7%), so leaderboard rank does not predict corpus name recall. MinerU point is coverage-contaminated but the conclusion holds without it."),
  'H151': ('PARTIALLY CONFIRMED',
    f"Clause (a) holds - {rep['H151']['table_region_frac']:.1%} of losses lie in table regions (>=70% bar) - but clause (b) is refuted: pypdf, a plain text extractor with no table-structure stage, is the single best recoverer (93.7%) and clears H146's bar, so recovery is not >=2x driven by table capability. The differentiator is pymupdf4llm's specific cell-merge bug, not the presence of a table stage. The loss set is defined such that the without-stage group recovers it near-fully by construction."),
}
for h,(v,s) in verdicts.items():
    print(f'{h} {v}\n    {s}\n')

H146 CONFIRMED
    Docling recovers 82/95 = 86.3% of the loss set at 25.3 s/page CPU, clearing the >=60% bar and beating the 75.8% union-of-three baseline.

H147 REFUTED
    The all-parser-absent SleepStyle family is a trademark-glyph normalization artifact - pypdf, docling and olmOCR each recover 7/7 under symbol-stripped matching, so vision is not necessary; MinerU2.5 itself is unmeasurable (VLM crashed on 10/11 chunks, 6/107 pages, 0 family pages covered).

H148 NOT MEASURABLE
    MinerU2.5 covered 0/18 numeric-floor pages (VLM crash) and dots.ocr was not run, so the >=10-point recall-gain-with-digit-precision bar cannot be evaluated on the registered engines; the olmOCR-off vision proxy lifted 0/16 of the all-text-parser-failed gold residue, i.e. no vision lift observed.

H149 CONFIRMED
    Anchored olmOCR recovers 77/95 = 81.1% (>=half) and anchor-off recovers strictly more at 83/95 = 87.4% - document-anchoring is a liability on born-digital pages.

H150 CONFIRMED
    Spearman(TED

In [4]:
# Assemble and write the final combined report
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d-%H%M%S')
final = {
  'round': 'R14 parser/OCR bake-off (H146-H151)',
  'generated_utc': stamp,
  'scoring': 'CPU only; reuses parser_harness.py + score_round.py cached outputs',
  'loss_set_n': len(S.LOSS),
  'severed_table_rows_h144': severed_rows,
  'loss_recovery_by_parser': recall_by_parser,
  'preservation_by_parser': preservation_by_parser,
  'coverage': cov,
  'verdicts': {h: {'verdict': v, 'statement': s} for h,(v,s) in verdicts.items()},
  'per_hypothesis_reports': {h: rep[h] for h in rep},
  'parser_recommendation': {
     'ship': 'text-layer swap, not vision',
     'pypdf': {'loss_recovery': recall_by_parser['pypdf'], 'preservation': preservation_by_parser['pypdf'],
                'runtime': '17.9s / 517 pages (~0.03 s/page)', 'gpu': False},
     'docling': {'loss_recovery': recall_by_parser['docling'], 'preservation': preservation_by_parser['docling'],
                  'runtime_s_per_page': rep['H146']['runtime_s_per_page'], 'gpu': False,
                  'note': 'adds TableFormer table-structure stage for H152/H153 row-records'},
     'olmocr_off': {'loss_recovery': recall_by_parser['olmocr_off'], 'gpu': True, 'note': 'page-scoped; does not beat pypdf on recovery'},
     'mineru2.5': {'status': 'inoperable on this environment - VLM tensor-shape crash on 10/11 chunks'},
     'decision': 'Ship pymupdf4llm union pypdf (93.7% recovery, ~0.03 s/page, CPU) or adopt Docling as the backend (86.3% + table stage). Vision is not justified.',
  },
  'open_questions': [
     'MinerU2.5 VLM tensor-shape crash (env/version bug) leaves H147/H148 vision-necessity formally untested on MinerU/dots.ocr.',
     'Does any vision engine lift the 16-pair all-text-parser-failed numeric-floor residue? olmOCR-off did not; MinerU/dots.ocr untested.',
     'olmOCR/MinerU whole-doc preservation is not comparable to text parsers (page-scoped runs); a full-doc olmOCR pass would be needed to compare preservation fairly.',
  ],
  'promotion_candidates': [
     'Docling backend swap (H146 CONFIRMED) into the ingest pipeline as the table-structure-stage parser feeding H152 row-records.',
     'pypdf as the cheap union partner for pymupdf4llm (93.7% recovery, fastest, CPU).',
     'H51 name-recall harness adopted as the standing acceptance test for any future parser change (H150 registered outcome).',
  ],
}
out = ROOT / 'reports' / f'parser-round-final-{stamp}.json'
out.write_text(json.dumps(final, indent=2))
print('wrote', out)
print(json.dumps({h: final['verdicts'][h]['verdict'] for h in final['verdicts']}, indent=2))

wrote /home/lab/workspace/learning/projects/knowledge-graph-foundry/reports/parser-round-final-20260707-135106.json
{
  "H146": "CONFIRMED",
  "H147": "REFUTED",
  "H148": "NOT MEASURABLE",
  "H149": "CONFIRMED",
  "H150": "CONFIRMED",
  "H151": "PARTIALLY CONFIRMED"
}
